# Proyek Akhir: Menyelesaikan Permasalahan Institusi Pendidikan
## Analisis Data Siswa dan Prediksi Dropout

**Nama:** [Nama Anda]
**Batch:** [Batch Anda]
**Tanggal:** 2026-03-24

## 1. Business Understanding

### Latar Belakang
Jaya Jaya Institut merupakan salah satu institusi pendidikan perguruan yang telah berdiri sejak tahun 2000. Hingga saat ini ia telah mencetak banyak lulusan dengan reputasi yang sangat baik. Akan tetapi, terdapat banyak juga siswa yang tidak menyelesaikan pendidikannya alias dropout.

Jumlah dropout yang tinggi ini tentunya menjadi salah satu masalah yang besar untuk sebuah institusi pendidikan. Oleh karena itu, Jaya Jaya Institut ingin mendeteksi secepat mungkin siswa yang mungkin akan melakukan dropout sehingga dapat diberi bimbingan khusus.

### Permasalahan Bisnis
1. Bagaimana mengidentifikasi siswa yang berisiko tinggi melakukan dropout?
2. Faktor-faktor apa saja yang paling berpengaruh terhadap dropout siswa?
3. Bagaimana membuat sistem prediksi yang akurat untuk mendeteksi siswa yang berisiko dropout?

### Tujuan
1. Menganalisis faktor-faktor yang mempengaruhi dropout siswa
2. Membuat model machine learning untuk memprediksi siswa yang berisiko dropout
3. Membuat dashboard untuk memonitor performa siswa secara real-time

### Cakupan Proyek
- Menggunakan dataset Students Performance dari Dicoding Academy
- Melakukan analisis deskriptif terhadap data
- Membuat model prediksi multi-class (Dropout, Enrolled, Graduate)
- Membuat dashboard interaktif menggunakan Streamlit
- Membuat dashboard analisis bisnis menggunakan Metabase

## 2. Data Understanding

Dataset yang digunakan adalah Students Performance Dataset dari Dicoding Academy yang berisi informasi tentang 4,424 siswa perguruan tinggi.

### Sumber Data
- `students_performance.csv` - Dataset mentah (raw data) dengan 34 fitur
- `students_performance_processed.csv` - Dataset yang sudah diproses untuk permodelan

### Deskripsi Dataset
Dataset ini berisi informasi tentang siswa perguruan tinggi dengan fitur-fitur seperti:
- Informasi pribadi (Marital_status, Gender, Age_at_enrollment, dll)
- Informasi akademik (Admission_grade, Curricular_units, dll)
- Informasi ekonomi (Unemployment_rate, Inflation_rate, GDP)
- Status akhir (Target variable: Dropout, Enrolled, Graduate)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
sns.set_palette('Set2')
%matplotlib inline

In [ ]:
# Load raw dataset untuk Exploratory Data Analysis (EDA)
df_raw = pd.read_csv('students_performance.csv', sep=';')

print(f'Jumlah baris: {df_raw.shape[0]}')
print(f'Jumlah kolom: {df_raw.shape[1]}')
df_raw.head()

In [ ]:
# Informasi dataset mentah
df_raw.info()

In [ ]:
# Distribusi Status pada data mentah
print('Distribusi Status:')
print(df_raw['Status'].value_counts())
print('\nPersentase Status:')
print(df_raw['Status'].value_counts(normalize=True) * 100)

## 3. Data Preparation

### 3.1 Data Exploration

Melakukan eksplorasi data untuk memahami karakteristik dataset.

In [ ]:
# Statistik deskriptif
df_raw.describe()

In [ ]:
# Cek missing values
df_raw.isnull().sum()

In [ ]:
# Visualisasi distribusi Status
plt.figure(figsize=(8, 6))
df_raw['Status'].value_counts().plot(kind='bar')
plt.title('Distribusi Status Siswa')
plt.xlabel('Status')
plt.ylabel('Jumlah')
plt.xticks(rotation=0)
plt.show()

### 3.2 Data Preprocessing

Melakukan preprocessing data untuk menyiapkan data bagi model machine learning.

In [ ]:
# Load processed dataset
df_processed = pd.read_csv('students_performance_processed.csv')

print(f'Jumlah baris: {df_processed.shape[0]}')
print(f'Jumlah kolom: {df_processed.shape[1]}')
df_processed.head()

In [ ]:
# Cek missing values pada dataset processed
df_processed.isnull().sum()

In [ ]:
# Split data menjadi features dan target
X = df_processed.drop('Status', axis=1)
y = df_processed['Status']

print(f'Features: {X.shape}')
print(f'Target: {y.shape}')

In [ ]:
# Encode target variable
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

print('Kelas target:', le_target.classes_)
print('Encoded target:', np.unique(y_encoded))

In [ ]:
# Split data menjadi train dan test set
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print(f'Train set: {X_train.shape}, {y_train.shape}')
print(f'Test set: {X_test.shape}, {y_test.shape}')

## 4. Model Development

### 4.1 Model Training

Membuat dan melatih model machine learning.

In [ ]:
# Inisialisasi model Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

# Latih model
rf_model.fit(X_train, y_train)

print('Model berhasil dilatih!')

In [ ]:
# Prediksi pada test set
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)

### 4.2 Model Evaluation

Mengevaluasi performa model.

In [ ]:
# Hitung accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f}')

In [ ]:
# Classification report
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=le_target.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le_target.classes_, yticklabels=le_target.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# ROC AUC Score (multi-class)
auc_score = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
print(f'ROC AUC Score: {auc_score:.4f}')

### 4.3 Feature Importance

Melihat fitur-fitur yang paling berpengaruh terhadap prediksi.

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print('Top 10 Feature Importance:')
print(feature_importance.head(10))

In [ ]:
# Visualisasi feature importance
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(10)
plt.barh(range(len(top_features)), top_features['importance'].values)
plt.yticks(range(len(top_features)), top_features['feature'].values)
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Top 10 Feature Importance')
plt.gca().invert_yaxis()
plt.show()

## 5. Model Deployment

### 5.1 Save Model

Menyimpan model yang telah dilatih.

In [ ]:
import joblib

# Save model
joblib.dump(rf_model, 'model.pkl')
joblib.dump(le_target, 'label_encoder.pkl')

print('Model dan label encoder berhasil disimpan!')

### 5.2 Model Testing

Menguji model dengan data baru.

In [ ]:
# Load model
loaded_model = joblib.load('model.pkl')
loaded_encoder = joblib.load('label_encoder.pkl')

# Test prediksi
sample_data = X_test.iloc[0:1]
prediction = loaded_model.predict(sample_data)
prediction_proba = loaded_model.predict_proba(sample_data)

print(f'Prediksi: {loaded_encoder.inverse_transform(prediction)[0]}')
print(f'Probabilitas: {dict(zip(loaded_encoder.classes_, prediction_proba[0]))}')

## 6. Conclusion

### Kesimpulan
1. Model Random Forest berhasil mencapai akurasi sebesar {accuracy:.2%} dalam memprediksi status siswa
2. ROC AUC score sebesar {auc_score:.4f} menunjukkan performa model yang baik
3. Fitur-fitur terpenting yang mempengaruhi dropout adalah: {', '.join(feature_importance.head(5)['feature'].tolist())}
4. Model dapat digunakan untuk mengidentifikasi siswa yang berisiko dropout sejak dini

### Rekomendasi
1. Implementasikan model ini ke dalam sistem informasi akademik
2. Lakukan monitoring performa model secara berkala
3. Kembangkan model dengan data yang lebih banyak dan fitur yang lebih lengkap
4. Buat sistem notifikasi untuk staf akademik ketika ada siswa yang berisiko tinggi

## 7. References

1. Dicoding Academy. (2026). Belajar Penerapan Data Science.
2. Students Performance Dataset: https://github.com/dicodingacademy/dicoding_dataset
3. Scikit-learn Documentation: https://scikit-learn.org
4. Pandas Documentation: https://pandas.pydata.org